# Extract depth maps from video
This example script extracts depth maps from depth videos. First, load the required libraries.

In [2]:
import cv2
import pandas as pd
from pathlib import Path

Then input the path to the depth video and the corresponding timestamp file.

In [ ]:
# load depth video
depth_video = "" # path to the dataset/DEPTH/DEPTH.avi
depth_timestamp_file = "" # path to the dataset/DEPTH/DEPTH_timestamp.avi
depth_timestamp = pd.read_csv(depth_timestamp_file)

Then set the output dir to save the depth maps.

In [ ]:
# save path
output_dir = Path("") # path to save the depth frames
output_dir.mkdir(parents=True, exist_ok=True)

Once the input and output path are ready, the following code can extract the depth map from the depth video and name the depth map with its global timestamp.

In [ ]:
# start converting, the saved depth maps are in uint16
cap = cv2.VideoCapture(
    depth_video,
    cv2.CAP_FFMPEG,
    [cv2.CAP_PROP_CONVERT_RGB, 0],
)

if not cap.isOpened():
    raise RuntimeError(f"Cannot open video: {depth_video}")
frame_idx = 0

while True:
    success, frame = cap.read()

    if not success:
        break

    if frame.dtype != "uint16":
        raise TypeError(
            f"Expected uint16 depth, but OpenCV returned {frame.dtype} "
            f"with shape {frame.shape}"
        )

    frame_ts = depth_timestamp.loc[frame_idx, "timestamp"]
    output_path = output_dir / f"{frame_ts:.06f}.tiff"

    if not cv2.imwrite(str(output_path), frame):
        raise RuntimeError(f"Failed to save: {output_path}")

    frame_idx += 1

cap.release()